In [149]:
import warnings

# Suppress DeprecationWarning messages from jupyter_client
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module='jupyter_client'
)

In [150]:
!pip install -q \
langgraph \
langchain \
langchain-community \
langchain-openai \
chromadb \
duckduckgo-search \
sentence-transformers \
langchain-huggingface \
python-dotenv \
faiss-cpu

In [151]:
import os

print(os.listdir("/content"))

['.config', 'Adaptive-Tech-Support', 'requirements.txt', 'test_agents.py', 'retriever.py', 'orion_hub_manual .txt', 'chroma_db', '.ipynb_checkpoints', 'drive', '.env', 'prompts.py', 'state.py', 'app.py', '__pycache__', 'agents.py', 'test_graph.py', 'graph.py', 'untitled1', 'sample_data']


In [152]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/orion_hub_manual .txt")

docs = loader.load()

print(docs[0].page_content[:500])

﻿======================================================================
  ENTERPRISE TECHNICAL MANUAL & FAQ: ORION SMARTHUB X1 PRO GATEWAY
Product Engine: Orion Core Ecosystem
Hardware Model: SmartHub-X1-PRO
Firmware Baseline: Stable Build v4.12.0
Document ID: TECH-REF-2026-REV2


1. HARDWARE SPECIFICATIONS & INTERNAL CHIPSETS
----------------------------------------------------------------------
The Orion SmartHub X1 Pro ser


In [153]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

splits = splitter.split_documents(docs)

print("Number of Chunks:", len(splits))
print("\nFirst Chunk:\n")
print(splits[0].page_content)

Number of Chunks: 12

First Chunk:

﻿======================================================================
  ENTERPRISE TECHNICAL MANUAL & FAQ: ORION SMARTHUB X1 PRO GATEWAY
Product Engine: Orion Core Ecosystem
Hardware Model: SmartHub-X1-PRO
Firmware Baseline: Stable Build v4.12.0
Document ID: TECH-REF-2026-REV2


1. HARDWARE SPECIFICATIONS & INTERNAL CHIPSETS
----------------------------------------------------------------------
The Orion SmartHub X1 Pro serves as a high-performance local bridge 
and supervisor node for hybrid smart home fabrics.


In [154]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded Successfully


In [155]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    splits,
    embedding
)

retriever = db.as_retriever(search_kwargs={"k":3})

print("Vector Database Created Successfully")

Vector Database Created Successfully


In [156]:
query = "How do I factory reset the hub?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 50)
    print(doc.page_content)


Result 1
--------------------------------------------------
4. SYSTEM RESTORATION & HARD RESET ARCHITECTURE
----------------------------------------------------------------------
- Soft System Reboot: Log into the cloud dashboard panel, navigate to 
  System Admin Console > Maintenance Tools, and commit the "Restart Hub" 
  pipeline. Hardware method: Disconnect the power coupling from the 
  USB-C terminal port for exactly 15 seconds, then re-insert.
  
- Hard Factory Reset: Locate the recessed sub-surface microswitch button 
  labeled "RESET" on the rear panel assembly (situated directly to the

Result 2
--------------------------------------------------
4. SYSTEM RESTORATION & HARD RESET ARCHITECTURE
----------------------------------------------------------------------
- Soft System Reboot: Log into the cloud dashboard panel, navigate to 
  System Admin Console > Maintenance Tools, and commit the "Restart Hub" 
  pipeline. Hardware method: Disconnect the power coupling from the 
  

In [157]:
!pip install -q langchain-google-genai

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Say Hello")

print(response.content)

In [ ]:
rewrite_prompt = """
You are a technical support expert.
Rewrite the user's question into a keyword-rich search query.
Only return the rewritten query.

Question:
{question}
"""

In [ ]:
def rewrite_query(question):

    prompt = rewrite_prompt.format(question=question)
    response = llm.invoke(prompt)
    return response.content

question = "Hub blinking red"
optimized = rewrite_query(question)
print(optimized)

In [ ]:
grading_prompt = """
You are a QA Engineer.
Your job is to determine whether the retrieved documents are relevant to the user's question.
Return ONLY JSON.

Example:

{{
    "relevance_score":"yes"
}}

or

{{
    "relevance_score":"no"
}}

Question:
{question}

Retrieved Documents:
{documents}
"""

In [ ]:
import json

def grade_documents(question, documents):

    docs = "\n\n".join([doc.page_content for doc in documents])

    prompt = grading_prompt.format(
        question=question,
        documents=docs
    )

    response = llm.invoke(prompt)

    print("LLM Response:")
    print(response.content)

    try:
        result = json.loads(response.content)
        return result["relevance_score"]
    except Exception:
        return "no"

In [ ]:
question = "How do I reset the hub?"

documents = retriever.invoke(question)

score = grade_documents(question, documents)

print("\nRelevance Score:", score)

In [ ]:
question = "Who won the FIFA World Cup?"

documents = retriever.invoke(question)

score = grade_documents(question, documents)

print(score)

In [ ]:
question = "Who won the FIFA World Cup?"

documents = retriever.invoke(question)

score = grade_documents(question, documents)

if score == "yes":
    print("Use Local Documents")
else:
    print("Go to Web Search")

                    Retrieve Documents
                            |
                        QA Agent
                            |
                        Relevant
                       |       |
      YES(Generate Answer)     NO(Web Search)





In [ ]:
!pip install -q ddgs

In [ ]:
from ddgs import DDGS

def web_search(query):

    results = []

    with DDGS() as ddgs:
        search_results = ddgs.text(query, max_results=3)

        for result in search_results:
            results.append(result["body"])

    return results

In [ ]:
query = "What is ChatGPT?"

results = web_search(query)

for i, r in enumerate(results, 1):
    print(f"\nResult {i}")
    print("-"*50)
    print(r)

In [ ]:
question = "How do I reset the hub?"

documents = retriever.invoke(question)

score = grade_documents(question, documents)

print("QA Score:", score)

if score == "yes":
    print("\nUsing Local Documents\n")
    final_docs = [doc.page_content for doc in documents]

else:
    print("\nUsing Web Search\n")
    final_docs = web_search(question)

for d in final_docs:
    print("-"*40)
    print(d)

In [ ]:
generate_prompt = """
You are an expert technical support engineer.

Answer the user's question using ONLY the provided context.

If the answer cannot be found, say:

"I don't have enough information."

Context:

{context}

Question:

{question}

Answer:
"""

In [ ]:
def generate_answer(question, documents):

    context = "\n\n".join(documents)

    prompt = generate_prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return response.content

In [ ]:
question = "How do I reset the hub?"

documents = retriever.invoke(question)

docs = [doc.page_content for doc in documents]

answer = generate_answer(question, docs)

print(answer)

In [ ]:
question = "What is ChatGPT?"

web_docs = web_search(question)

answer = generate_answer(question, web_docs)

print(answer)

In [ ]:
hallucination_prompt = """
You are a QA Engineer.

Compare the generated answer with the provided documents.
If the answer is completely supported by the documents return ONLY:

{{
"hallucination_check":"passed"
}}

Otherwise return ONLY:

{{
"hallucination_check":"failed"
}}

Documents:

{documents}

Generated Answer:

{answer}
"""

In [ ]:
import json

def check_hallucination(documents, answer):

    docs = "\n\n".join(documents)

    prompt = hallucination_prompt.format(
        documents=docs,
        answer=answer
    )

    response = llm.invoke(prompt)

    print(response.content)

    try:
        result = json.loads(response.content)
        return result["hallucination_check"]

    except:
        return "failed"

In [ ]:
question = "How do I reset the hub?"

documents = retriever.invoke(question)

docs = [doc.page_content for doc in documents]

answer = generate_answer(question, docs)

print(answer)

print()

check = check_hallucination(docs, answer)

print(check)

In [ ]:
fake_answer = """
Press the reset button for 30 seconds.
"""

documents = retriever.invoke("How do I reset the hub?")

docs = [doc.page_content for doc in documents]

check = check_hallucination(docs, fake_answer)

print(check)

In [ ]:
!pip install -q langgraph

In [ ]:
from typing import TypedDict, List

from langgraph.graph import StateGraph, END

In [ ]:
class GraphState(TypedDict):

    original_query: str
    optimized_query: str
    documents: List[str]
    generation: str
    relevance_score: str
    hallucination_check: str
    loop_count: int

In [ ]:
def rewrite_node(state):

    state["optimized_query"] = rewrite_query(
        state["original_query"]
    )

    return state

In [ ]:
def retrieve_node(state):

    state["documents"] = retriever.invoke(
        state["optimized_query"]
    )

    return state

In [ ]:
def grade_node(state):

    state["relevance_score"] = grade_documents(
        state["original_query"],
        state["documents"]
    )

    return state

In [ ]:
def web_search_node(state):
    state["documents"] = web_search(
        state["optimized_query"]
    )

    return state

In [ ]:
def generate_node(state):
    state["generation"] = generate_answer(
        state["original_query"],
        state["documents"]
    )

    return state

In [ ]:
def hallucination_node(state):
    state["hallucination_check"] = check_hallucination(
        state["documents"],
        state["generation"]
    )

    return state

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("rewrite", rewrite_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("websearch", web_search_node)
workflow.add_node("generate", generate_node)
workflow.add_node("hallucination", hallucination_node)

In [ ]:
def grade_documents_cond(state):
    if state["relevance_score"] == "yes":
        return "generate"
    else:
        return "websearch"

In [ ]:
def generate_answer_cond(state):
    if state["hallucination_check"] == "passed":
        return END
    else:
        return "websearch"

In [ ]:
workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    grade_documents_cond
)
workflow.add_edge("websearch", "generate")
workflow.add_edge("generate", "hallucination")
workflow.add_conditional_edges(
    "hallucination",
    generate_answer_cond
)

app = workflow.compile()

In [ ]:
inputs = {"original_query": "How do I factory reset the hub?"}
response = app.invoke(inputs)

print("Final Answer:")
print(response["generation"])

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("rewrite", rewrite_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("websearch", web_search_node)
workflow.add_node("generate", generate_node)
workflow.add_node("hallucination", hallucination_node)

In [ ]:
def grade_documents_cond(state):
    if state["relevance_score"] == "yes":
        return "generate"
    else:
        return "websearch"

In [ ]:
def generate_answer_cond(state):
    if state["hallucination_check"] == "passed":
        return END
    else:
        # This could be a loop back to retrieve with a refined query
        # or to websearch depending on the desired retry logic.
        # For now, let's end for simplicity or could loop to rewrite or websearch.
        return "websearch"

In [ ]:
workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    grade_documents_cond
)
workflow.add_edge("websearch", "generate")
workflow.add_edge("generate", "hallucination")
workflow.add_conditional_edges(
    "hallucination",
    generate_answer_cond
)

app = workflow.compile()

In [ ]:
inputs = {"original_query": "How do I factory reset the hub?"}
response = app.invoke(inputs)

print("Final Answer:")
print(response["generation"])

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("rewrite", rewrite_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("websearch", web_search_node)
workflow.add_node("generate", generate_node)
workflow.add_node("hallucination", hallucination_node)

In [ ]:
def grade_documents_cond(state):
    if state["relevance_score"] == "yes":
        return "generate"
    else:
        return "websearch"

In [ ]:
def generate_answer_cond(state):
    if state["hallucination_check"] == "passed":
        return END
    else:
        return "websearch"

In [ ]:
workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    grade_documents_cond
)
workflow.add_edge("websearch", "generate")
workflow.add_edge("generate", "hallucination")
workflow.add_conditional_edges(
    "hallucination",
    generate_answer_cond
)

app = workflow.compile()

In [ ]:
inputs = {"original_query": "How do I factory reset the hub?"}
response = app.invoke(inputs)

print("Final Answer:")
print(response["generation"])

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("rewrite", rewrite_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("websearch", web_search_node)
workflow.add_node("generate", generate_node)
workflow.add_node("hallucination", hallucination_node)

In [ ]:
def grade_documents_cond(state):
    if state["relevance_score"] == "yes":
        return "generate"
    else:
        return "websearch"

In [ ]:
def generate_answer_cond(state):
    if state["hallucination_check"] == "passed":
        return END
    else:
        # This could be a loop back to retrieve with a refined query
        # or to websearch depending on the desired retry logic.
        # For now, let's end for simplicity or could loop to rewrite or websearch.
        return "websearch"

In [ ]:
workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    grade_documents_cond
)
workflow.add_edge("websearch", "generate")
workflow.add_edge("generate", "hallucination")
workflow.add_conditional_edges(
    "hallucination",
    generate_answer_cond
)

app = workflow.compile()

In [ ]:
workflow.set_entry_point("rewrite")
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    grade_documents_cond
)
workflow.add_edge("websearch", "generate")
workflow.add_edge("generate", "hallucination")
workflow.add_conditional_edges(
    "hallucination",
    generate_answer_cond
)

app = workflow.compile()

In [ ]:
graph = workflow.compile()

In [ ]:
inputs = {

    "original_query":"How do I reset the hub?",

    "optimized_query":"",

    "documents":[],

    "generation":"",

    "relevance_score":"",

    "hallucination_check":"",

    "loop_count":0
}

result = graph.invoke(inputs)

print(result["generation"])

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
!python retriever.py

In [ ]:
!python test_agents.py

In [ ]:
!pip install -q streamlit